In [120]:
import pandas as pd
import os
import random
import numpy as np
import json
import shelve
import math

### Common 

In [2]:
sample_count = 1000

In [3]:
np.random.seed(42)

In [63]:
def apply_condition(data, col, operator, value):
    if operator == '>':
        return np.where(data[col] > value, 1, 0)
    elif operator == '<':
        return np.where(data[col] < value, 1, 0)
    elif operator == '>=':
        return np.where(data[col] >= value, 1, 0)
    elif operator == '<=':
        return np.where(data[col] <= value, 1, 0)
    elif operator == '=':
        return np.where(data[col] == value, 1, 0)
    elif operator == '!=':
        return np.where(data[col] != value, 1, 0)
    else:
        raise ValueError("Unsupported operator")

In [7]:
def read_json(file_path):
    with open(file_path, 'r', encoding='utf-8') as f:
        return json.load(f)

In [8]:
value_counts = read_json("/remote-home/cs_acmis_wsf/LCEModels/attention-tree-lstm/data/value_count.json")

In [252]:
metadata = {}
with shelve.open(f'/remote-home/cs_acmis_wsf/LCEModels/attention-tree-lstm/data/metadata/metadata-incremental.db') as db:
    metadata["tables"] = db["tables"]
    metadata["tables_row"] = db["tables_row"]
    metadata["columns"] = db["columns"]

In [253]:
meta_columns = metadata["columns"]

In [50]:
def getColType(key):
    return meta_columns[key]["type"]

In [57]:
def getColValues(key):
    return meta_columns[key]["value"]

In [125]:
def getOperators(lx):
    if lx =='VARCHAR' or lx =='BOOL':
        return ["!=","="]
    else:
        return ["<","<=","!=","=",">",">="]

In [243]:
def varcharCondition(data,col,operators,values):
    col_bitmap = {}
    for o in operators:
        for v in values:
            key = col+o+"'"+str(v)+"'"
            bitmap = apply_condition(data,col,o,v)
            col_bitmap[key] = bitmap
    return col_bitmap

In [144]:
def floatCondition(data,col,operators,values):
    col_bitmap = {}
    min_val = float(values[0])
    max_value = float(values[1])
    diff = max_value - min_val
    num = 2+2* math.ceil(diff / 10.0)
    random_numbers = np.linspace(min_val, max_val, num)
    for i in range(len(random_numbers)):
        if i ==0:
            op = operators[2:]
        elif i==len(random_numbers)-1:
            op = operators[:4]
        else:
            op = operators
        for o in op:
            key = col+o+str(random_numbers[i])
            bitmap = apply_condition(data,col,o,random_numbers[i])
            col_bitmap[key] = bitmap
    return col_bitmap

In [171]:
def intCondition(data,col,operators,values):
    col_bitmap = {}
    min_val = int(values[0])
    max_val = int(values[1])
    diff = max_val - min_val
    num = 2+2* math.ceil(diff / 10)
    random_numbers = np.unique(np.round(np.linspace(min_val, max_val, num)).astype(int))
    for i in range(len(random_numbers)):
        if i ==0:
            op = operators[2:]
        elif i==len(random_numbers)-1:
            op = operators[:4]
        else:
            op = operators
        for o in op:
            key = col+o+str(random_numbers[i])
            bitmap = apply_condition(data,col,o,random_numbers[i])
            col_bitmap[key] = bitmap
    return col_bitmap

In [222]:
def boolCondition(data,col,operators,values):
    col_bitmap = {}
    for v in [True,False]:
            key = col.replace('reads','read')+" = "+str(v).replace('True','TRUE').replace('False','FALSE')
            bitmap = apply_condition(data,col,'=',v)
            col_bitmap[key] = bitmap
    return col_bitmap

### business

In [199]:
table = 'business'

In [200]:
business_vs = value_counts[table]

In [201]:
cols = list(business_vs.keys())

In [4]:
business = pd.read_csv('/remote-home/cs_acmis_wsf/ai4dingo/yelp/vector/business_mf.csv')

In [5]:
sampled_business = business.sample(n=sample_count)

In [204]:
business = {}
for col in cols:
    key = table+"."+col
    lx = getColType(key)
    operators = getOperators(lx)
    values = getColValues(key)
    if lx == 'VARCHAR':
        col_bitmap = varcharCondition(sampled_business,col,operators,values)
        business[col] = col_bitmap
    elif lx =='FLOAT':
        col_bitmap = floatCondition(sampled_business,col,operators,values)
        business[col] = col_bitmap
    elif lx == 'INT':
        col_bitmap = intCondition(sampled_business,col,operators,values)
        business[col] = col_bitmap
    else:
        col_bitmap = boolCondition(sampled_business,col,operators,values)
        business[col] = col_bitmap

### review

In [185]:
base_directory = '/remote-home/cs_acmis_wsf/ai4dingo/yelp/vector/split_file/review'
base_filename = 'part_'
file_extension = '.csv'

In [186]:
dfs = []

# 使用循环读取part_1.csv到part_40.csv文件
for i in range(1, 41):  # 从1到40，包含40
    file_name = f"{base_filename}{i}{file_extension}"  # 构建文件名
    file_path = os.path.join(base_directory, file_name)  # 构建完整的文件路径
    df = pd.read_csv(file_path)  # 读取CSV文件
    dfs.append(df)  # 将DataFrame添加到列表中

In [187]:
review = pd.concat(dfs, ignore_index=True)

In [188]:
sampled_review = review.sample(n=sample_count)

In [189]:
sampled_review.head(5)

,review_id,stars,useful,funny,cool,likes,dislikes,views,reads,text,feature
2518411,_jKgiQZ2wv4L8gayPkLEQA,5.0,0.0,0.0,0.0,100,30,18,True,The best quality hole in the wall restaurant. ...,"[0.524851381778717, 0.41564345359802246, -0.13..."
1978597,jqZeEx0pRu6qzN-uxAVhBQ,2.0,2.0,0.0,0.0,9,9,2,False,"Aside from the head pharmacist, the pharmacy s...","[0.5132831335067749, 0.4344465136528015, 0.047..."
243823,IMJlmiTzTRtEEjfQUsdHRw,5.0,1.0,0.0,0.0,121,46,15,True,Sabrina introduced herself and was super frien...,"[0.4917669892311096, 0.35488027334213257, 0.05..."
118715,ivvhzF7l6oFT23Pu9qlh2w,5.0,0.0,0.0,0.0,139,32,13,True,Rachel Cajda is amazing! I have thin hair and...,"[0.5320309996604919, 0.33023738861083984, 0.11..."
1341685,F2h_w-F7lAXK3kLWLJZAkw,4.0,0.0,0.0,0.0,107,25,11,False,Micheal (the Mixologist) was a very good resou...,"[0.6624698638916016, 0.2440883368253708, 0.225..."


In [206]:
table = 'review'

In [207]:
review_vs = value_counts[table]

In [208]:
cols = list(review_vs.keys())

In [209]:
cols

['stars', 'useful', 'funny', 'cool', 'likes', 'dislikes', 'views', 'read']

In [210]:
print(getColType(table+"."+"read"))

BOOL


In [223]:
review = {}

In [224]:
for col in cols:
    key = table+"."+col
    lx = getColType(key)
    operators = getOperators(lx)
    values = getColValues(key)
    if lx == 'VARCHAR':
        col_bitmap = varcharCondition(sampled_review,col,operators,values)
        review[col] = col_bitmap
    elif lx =='FLOAT':
        col_bitmap = floatCondition(sampled_review,col,operators,values)
        review[col] = col_bitmap
    elif lx == 'INT':
        col_bitmap = intCondition(sampled_review,col,operators,values)
        review[col] = col_bitmap
    else:
        col = col.replace('read','reads')
        col_bitmap = boolCondition(sampled_review,col,operators,values)
        review[col] = col_bitmap

### tip

In [226]:
base_directory = '/remote-home/cs_acmis_wsf/ai4dingo/yelp/vector/split_file/tip'
base_filename = 'part_'
file_extension = '.csv'

In [227]:
dfs_1 = []

# 使用循环读取part_1.csv到part_10.csv文件
for i in range(1, 10):  # 从1到9，包含9
    file_name = f"{base_filename}{i}{file_extension}"  # 构建文件名
    file_path = os.path.join(base_directory, file_name)  # 构建完整的文件路径
    df = pd.read_csv(file_path)  # 读取CSV文件
    dfs_1.append(df)  # 将DataFrame添加到列表中

In [228]:
tip = pd.concat(dfs_1, ignore_index=True)

In [229]:
sampled_tip = tip.sample(n=sample_count)

In [230]:
table = 'tip'

In [231]:
tip_vs = value_counts[table]

In [232]:
cols = list(tip_vs.keys())

In [233]:
tip = {}

In [234]:
for col in cols:
    key = table+"."+col
    lx = getColType(key)
    operators = getOperators(lx)
    values = getColValues(key)
    if lx == 'VARCHAR':
        col_bitmap = varcharCondition(sampled_tip,col,operators,values)
        tip[col] = col_bitmap
    elif lx =='FLOAT':
        col_bitmap = floatCondition(sampled_tip,col,operators,values)
        tip[col] = col_bitmap
    elif lx == 'INT':
        col_bitmap = intCondition(sampled_tip,col,operators,values)
        tip[col] = col_bitmap
    else:
        col_bitmap = boolCondition(sampled_tip,col,operators,values)
        tip[col] = col_bitmap

###  problem

In [236]:
problem = pd.read_csv('/remote-home/cs_acmis_wsf/ai4dingo/mooccubex/csv/problem/problem.csv')

In [237]:
sampled_problem = problem.sample(n=sample_count)

In [238]:
table = 'problem'

In [239]:
problem_vs = value_counts[table]

In [240]:
cols = list(problem_vs.keys())

In [245]:
problem = {}

In [246]:
for col in cols:
    if col =='typetext':
        continue
    
    key = table+"."+col
    lx = getColType(key)
    operators = getOperators(lx)
    values = getColValues(key)
    if lx == 'VARCHAR':
        col_bitmap = varcharCondition(sampled_problem,col,operators,values)
        problem[col] = col_bitmap
    elif lx =='FLOAT':
        col_bitmap = floatCondition(sampled_problem,col,operators,values)
        problem[col] = col_bitmap
    elif lx == 'INT':
        col_bitmap = intCondition(sampled_problem,col,operators,values)
        problem[col] = col_bitmap
    else:
        col_bitmap = boolCondition(sampled_problem,col,operators,values)
        problem[col] = col_bitmap



### yelp_user

In [250]:
yelp_user = pd.read_csv('/remote-home/cs_acmis_wsf/ai4dingo/yelp/csv/user.csv')

In [254]:
sampled_yelp_user = yelp_user.sample(n=sample_count)

In [256]:
table = 'yelp_user'

In [257]:
yelp_user_vs = value_counts[table]

In [259]:
cols = list(yelp_user_vs .keys())

In [261]:
yelp_user = {}

In [262]:
for col in cols: 
    key = table+"."+col
    lx = getColType(key)
    operators = getOperators(lx)
    values = getColValues(key)
    if lx == 'VARCHAR':
        col_bitmap = varcharCondition(sampled_yelp_user,col,operators,values)
        yelp_user[col] = col_bitmap
    elif lx =='FLOAT':
        col_bitmap = floatCondition(sampled_yelp_user,col,operators,values)
        yelp_user[col] = col_bitmap
    elif lx == 'INT':
        col_bitmap = intCondition(sampled_yelp_user,col,operators,values)
        yelp_user[col] = col_bitmap
    else:
        col_bitmap = boolCondition(sampled_yelp_user,col,operators,values)
        yelp_user[col] = col_bitmap

### mcx_user

In [267]:
mcx_user = pd.read_csv('/remote-home/cs_acmis_wsf/ai4dingo/mooccubex/csv/mcx-user.csv')

In [270]:
sampled_mcx_user = mcx_user.sample(n=sample_count)

In [271]:
table = 'mcx_user'

In [272]:
mcx_user_vs = value_counts[table]

In [273]:
cols = list(mcx_user_vs .keys())

In [274]:
mcx_user = {}

In [275]:
for col in cols: 
    key = table+"."+col
    lx = getColType(key)
    operators = getOperators(lx)
    values = getColValues(key)
    if lx == 'VARCHAR':
        col_bitmap = varcharCondition(sampled_mcx_user,col,operators,values)
        mcx_user[col] = col_bitmap
    elif lx =='FLOAT':
        col_bitmap = floatCondition(sampled_mcx_user,col,operators,values)
        mcx_user[col] = col_bitmap
    elif lx == 'INT':
        col_bitmap = intCondition(sampled_mcx_user,col,operators,values)
        mcx_user[col] = col_bitmap
    else:
        col_bitmap = boolCondition(sampled_mcx_user,col,operators,values)
        mcx_user[col] = col_bitmap

In [277]:
large_bitmap = {
    "business":business,
    "review":review,
    "tip":tip,
    "problem":problem,
    "yelp_user":yelp_user,
    "mcx_user":mcx_user
}

In [289]:
with shelve.open('/remote-home/cs_acmis_wsf/LCEModels/attention-tree-lstm/data/bitmap/large_bitmap.db') as db:
    db['sample_bitmap'] = large_bitmap